# Fintech AFC Transactions and Data Quality Lab

Generate fictional transactions with the original 12-column schema. All normal dates range from 2025-01-01 through 2026-09-30. Names are plain client names without test labels, sequence suffixes, or synthetic prefixes. The company labels and fictional account numbers describe simulated transactions only.

Use Python 3.11 and keep client_registry.py next to this notebook. Both datasets import the same deterministic client registry. No real inputs or network access are needed. The code has been updated for fixed account ownership; existing output files have not been regenerated for this revision.

Transaction IDs, amounts, products, names, and accounts are protected from deliberate noise. Other eligible fields retain noise. OVERWRITE=True replaces existing outputs when you choose to run generation. CSV identifiers should be imported as text to preserve leading zeros.


## Schema, clients, and fixed account ownership

The original 12 columns and their order remain unchanged. The beneficiary name field is bene_clean_name; no bene_client_name column is added.

client_registry.py defines a shared one-client-to-many-accounts mapping. PAYPAL has 7 accounts, VENMO 3, PAYONEER 5, TTT MoneyCorp 7, WISE 5, and STRIPE 3. Non-fintech focal clients include AURORA GLOBAL TRADING LTD (3), CEDAR LOGISTICS GROUP (5), and QUARTZ IMPORT EXPORT LTD (7). These are explicit simulation settings. Focal business clients are eligible in the ordinary business pool.

All individuals, businesses, and banks also have stable names, entity IDs, and disjoint account sets of 3, 5, or 7 accounts. Names such as SAN ZHANG have no transaction-number suffix. An account belongs to exactly one client, regardless of transaction role or dataset. Each row selects a client first, then one of that client's accounts. Filtering either leg by a client's account set therefore identifies the same owner. The registry can be inspected using FOCAL_ACCOUNT_MAPPING and ACCOUNT_OWNERS.

Exactly 25% of rows (for a row count divisible by four) have one fintech leg and an ordinary individual/business counterparty. Other rows use ordinary clients, including non-fintech focal businesses. Names and accounts never receive noise or blank injection. Products are limited to Wires, Domestic ACH, and International ACH. Required amounts are positive Python floats; CSV stores numeric text and readers must parse it as float.

Only value_date, orig_cntry_code, and bene_cntry_code are eligible for noise and missing-value injection in this small schema. Date noise includes 99999999, 20260230, slash/hyphen formats, and blanks. Country noise includes 97, HH, ZZ, lowercase codes, and blanks. No city columns are added. Independently selected noise and missing-value rows can overlap. A blank is an empty CSV field.


In [ ]:
from pathlib import Path
from datetime import date, timedelta, datetime
from collections import Counter
import csv
import json
import os
import random
import re
import sqlite3
import tempfile
import time

ROW_COUNT = 1_000_000
SEED = 20260930
NOISE_RATE = 0.08
MISSING_ROW_RATE = 0.03  # Independent probability of selecting a row for blank injection.
MAX_MISSING_FIELDS = 3  # Select between one and this many existing fields per selected row.
OVERWRITE = True
WRITE_CLEAN_OUTPUTS = False
PROJECT_DIR = None  # Example: Path(r"C:\FCO-Review-main")
HEADERS = ["trxn_id", "lvl_2_product_txt", "amount", "value_date",
           "orig_raw_account_num", "bene_raw_account_num", "orig_raw_name", "bene_raw_name",
           "orig_clean_name", "bene_clean_name", "orig_cntry_code", "bene_cntry_code"]
if PROJECT_DIR is None:
    PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / "convert.py").is_file() and (p / "mapping.py").is_file()), None)
if PROJECT_DIR is None:
    raise RuntimeError("Set PROJECT_DIR to the project folder.")
PROJECT_DIR = Path(PROJECT_DIR).expanduser().resolve()
if not PROJECT_DIR.is_dir():
    raise ValueError("PROJECT_DIR must exist.")
if not isinstance(ROW_COUNT, int) or isinstance(ROW_COUNT, bool) or not 1 <= ROW_COUNT <= 9_999_999:
    raise ValueError("ROW_COUNT must be between 1 and 9,999,999.")
if not 0 <= NOISE_RATE <= 1:
    raise ValueError("NOISE_RATE must be between zero and one.")
if not 0 <= MISSING_ROW_RATE <= 1:
    raise ValueError("MISSING_ROW_RATE must be between zero and one.")
if not isinstance(MAX_MISSING_FIELDS, int) or isinstance(MAX_MISSING_FIELDS, bool) or not 1 <= MAX_MISSING_FIELDS <= len(HEADERS):
    raise ValueError("MAX_MISSING_FIELDS must be between one and the number of existing columns.")
OUTPUT_DIR = PROJECT_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "synthetic_wire_transactions_1m.csv"
REPORT_FILE = OUTPUT_DIR / "synthetic_afc_quality_report.json"
CLEAN_FILE = OUTPUT_DIR / "synthetic_afc_clean.csv"
QUARANTINE_FILE = OUTPUT_DIR / "synthetic_afc_quarantine.csv"
START_DATE = date(2025, 1, 1)
END_DATE = date(2026, 9, 30)
DATES = [(START_DATE + timedelta(days=i)).strftime("%Y%m%d")
         for i in range((END_DATE - START_DATE).days + 1)]
COUNTRIES = ("US", "GB", "DE", "FR", "CA", "AU", "JP", "SG", "NL", "CH", "SE", "NZ", "IN", "MX", "BR")
PRODUCTS = ("Wires", "Domestic ACH", "International ACH")
WEIGHTS = (40, 35, 25)
FINTECH_ROW_RATE = 0.25
NOISE_TYPES = ("date_sentinel", "date_impossible", "date_missing", "date_slash", "date_dash",
               "country_lowercase", "country_invalid",
               "country_numeric_97", "country_alpha_hh", "country_missing")
PROTECTED_COLUMNS = {0, 1, 2, 4, 5, 6, 7, 8, 9}
MISSING_ELIGIBLE_COLUMNS = tuple(i for i in range(len(HEADERS)) if i not in PROTECTED_COLUMNS)
print(f"Output: {OUTPUT_FILE}")
print(f"Records: {ROW_COUNT:,}; valid dates: {START_DATE} through {END_DATE}; noise rate: {NOISE_RATE:.1%}")


In [ ]:
import sys
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from client_registry import FINTECH_NAMES, build_registry, choose_pair

CLIENTS, ACCOUNT_OWNERS, CLIENT_POOLS = build_registry()
FINTECH_COMPANIES = tuple(name.upper() for name in FINTECH_NAMES)

# This table is the account ownership reference for either transaction leg.
FOCAL_ACCOUNT_MAPPING = {name: list(client["accounts"])
                         for name, client in CLIENTS.items() if client["focal"]}

def is_fintech_name(value):
    return value.strip().upper() in FINTECH_COMPANIES

def is_fintech_row(sequence):
    return (sequence - 1) % 4 == 0

def clean_record(rng, seq):
    product = rng.choices(PRODUCTS, weights=WEIGHTS, k=1)[0]
    orig, bene = choose_pair(rng, seq, product, CLIENTS, CLIENT_POOLS)
    day = rng.choice(DATES)
    amount = round(rng.uniform(1.00, 2_000_000.00), 2)
    return [f"{day}{rng.choice(('C', 'S'))}{seq:07d}", product, amount, day,
            rng.choice(orig["accounts"]), rng.choice(bene["accounts"]),
            orig["name"], bene["name"], orig["name"], bene["name"],
            orig["country"], bene["country"]]

def inject_noise(row, kind, noise_rng):
    if kind == "date_sentinel": row[3] = "99999999"
    elif kind == "date_impossible": row[3] = "20260230"
    elif kind == "date_missing": row[3] = ""
    elif kind in ("date_slash", "date_dash"):
        separator = "/" if kind == "date_slash" else "-"
        row[3] = separator.join((row[3][:4], row[3][4:6], row[3][6:]))
    elif kind in ("country_lowercase", "country_invalid", "country_numeric_97", "country_alpha_hh", "country_missing"):
        column = noise_rng.choice((10, 11))
        if kind == "country_lowercase": row[column] = row[column].lower()
        elif kind == "country_invalid": row[column] = "ZZ"
        elif kind == "country_numeric_97": row[column] = "97"
        elif kind == "country_alpha_hh": row[column] = "HH"
        else: row[column] = ""

def inject_missing(row, missing_rng):
    columns = missing_rng.sample(MISSING_ELIGIBLE_COLUMNS,
                                 min(missing_rng.randint(1, MAX_MISSING_FIELDS), len(MISSING_ELIGIBLE_COLUMNS)))
    newly_blank = [HEADERS[column] for column in columns if row[column] != ""]
    for column in columns: row[column] = ""
    return [HEADERS[column] for column in columns], newly_blank


## Generate raw transactions with deliberate noise

The filename is retained for compatibility; its contents now include multiple products. The transaction-ID suffix is unique before noise injection. No risk score or extra column is added to the CSV.


In [ ]:
targets = [OUTPUT_FILE, REPORT_FILE] + ([CLEAN_FILE, QUARANTINE_FILE] if WRITE_CLEAN_OUTPUTS else [])
if not OVERWRITE and any(path.exists() for path in targets):
    raise FileExistsError("An output exists. Set OVERWRITE=True to replace it.")
rng = random.Random(SEED)
noise_rng = random.Random(SEED + 1)
missing_rng = random.Random(SEED + 2)
missing_selected_fields = Counter()
newly_blank_fields = Counter()
generation_counts = Counter()
injected = Counter()
product_counts = Counter()
started = time.perf_counter()
descriptor, temp_name = tempfile.mkstemp(dir=OUTPUT_DIR, suffix=".csv.tmp")
os.close(descriptor)
temp_path = Path(temp_name)
try:
    with temp_path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(HEADERS)
        for seq in range(1, ROW_COUNT + 1):
            row = clean_record(rng, seq)
            product_counts[row[1]] += 1
            generation_counts["fintech_rows"] += int(is_fintech_name(row[8]) or is_fintech_name(row[9]))
            noise_selected = noise_rng.random() < NOISE_RATE
            if noise_selected:
                kind = noise_rng.choice(NOISE_TYPES)
                inject_noise(row, kind, noise_rng)
                injected[kind] += 1
            missing_selected = missing_rng.random() < MISSING_ROW_RATE
            if missing_selected:
                selected, newly_blank = inject_missing(row, missing_rng)
                missing_selected_fields.update(selected)
                newly_blank_fields.update(newly_blank)
                generation_counts["rows_selected_for_missing_injection"] += 1
            generation_counts["rows_selected_for_noise_or_missing"] += int(noise_selected or missing_selected)
            generation_counts["rows_selected_for_both"] += int(noise_selected and missing_selected)
            writer.writerow(row)
            if seq % 100_000 == 0: print(f"Written {seq:,} rows")
    os.replace(temp_path, OUTPUT_FILE)
finally:
    temp_path.unlink(missing_ok=True)
print(f"Generated {ROW_COUNT:,} rows in {time.perf_counter() - started:.1f}s")
print(f"Noise-injected rows: {sum(injected.values()):,}; file size: {OUTPUT_FILE.stat().st_size / 1024**2:.1f} MiB")
print("Injected noise counts:", dict(sorted(injected.items())))


## Normalization and ownership checks

The quality function preserves the raw CSV, normalizes unambiguous date/country formatting, and flags invalid required values. It checks both account/name pairs against the shared ownership registry. Names retain their configured spelling, including TTT MoneyCorp. A temporary SQLite index checks transaction-ID uniqueness during the saved-file audit.


In [ ]:
VALID_DATES = set(DATES)
VALID_COUNTRIES = set(COUNTRIES)
ID_PATTERN = re.compile(r"[0-9]{8}[CS][0-9]{7}")
ACCOUNT_PATTERN = re.compile(r"[0-9]{14}")

def normalize_record(original):
    if len(original) != len(HEADERS): return list(original), ["field_count"], []
    row = list(original)
    errors = []
    for i in range(len(HEADERS)):
        if isinstance(row[i], str): row[i] = row[i].strip()
    key = row[0]
    if not ID_PATTERN.fullmatch(key) or key[:8] not in VALID_DATES: errors.append("invalid_transaction_id")
    if row[1] not in PRODUCTS: errors.append("invalid_product")
    try:
        amount = float(row[2])
        if not (amount > 0 and amount < float("inf")): raise ValueError
        row[2] = f"{amount:.2f}"
    except (TypeError, ValueError, OverflowError):
        errors.append("invalid_amount")
    value = row[3]
    if re.fullmatch(r"[0-9]{4}([/-])[0-9]{2}\1[0-9]{2}", value):
        value = value.replace("/", "").replace("-", "")
    if value not in VALID_DATES: errors.append("invalid_date")
    else:
        row[3] = value
        if ID_PATTERN.fullmatch(key) and key[:8] != value: errors.append("id_date_mismatch")
    for i in (4, 5):
        if not ACCOUNT_PATTERN.fullmatch(row[i]): errors.append(HEADERS[i] + "_invalid")
    for raw_index, clean_index in ((6, 8), (7, 9)):
        row[raw_index] = " ".join(row[raw_index].split())
        row[clean_index] = " ".join(row[clean_index].split())
        if not row[raw_index] or not row[clean_index]: errors.append(HEADERS[clean_index] + "_missing")
    for account_index, raw_index, clean_index in ((4, 6, 8), (5, 7, 9)):
        owner = ACCOUNT_OWNERS.get(row[account_index])
        if owner is None or any(row[i] != owner["name"] for i in (raw_index, clean_index)):
            errors.append(HEADERS[account_index] + "_ownership_mismatch")
    fintech_legs = int(is_fintech_name(row[8])) + int(is_fintech_name(row[9]))
    if fintech_legs > 1: errors.append("both_legs_fintech")
    for i in (10, 11):
        row[i] = row[i].upper()
        if row[i] not in VALID_COUNTRIES: errors.append(HEADERS[i] + "_invalid")
    if all(row[i] in VALID_COUNTRIES for i in (10, 11)):
        if row[1] == "Domestic ACH" and row[10] != row[11]: errors.append("domestic_corridor_mismatch")
        if row[1] == "International ACH" and row[10] == row[11]: errors.append("international_corridor_mismatch")
    for i, value in enumerate(row):
        if value == "" and i not in PROTECTED_COLUMNS: errors.append(HEADERS[i] + "_missing")
    fixes = [HEADERS[i] for i in range(len(HEADERS)) if str(row[i]) != str(original[i])]
    return row, errors, fixes

reference = clean_record(random.Random(10), 1)
assert not normalize_record(reference)[1]
assert isinstance(reference[2], float) and reference[2] > 0
assert sum(is_fintech_name(reference[i]) for i in (8, 9)) == 1
for invalid in ("99999999", "20260230", "", "20241231", "20261001"):
    check = reference.copy(); check[3] = invalid
    assert "invalid_date" in normalize_record(check)[1]
for boundary in ("20250101", "20260930"):
    check = reference.copy(); check[3] = boundary; check[0] = boundary + check[0][8:]
    assert not normalize_record(check)[1]
for column in (10, 11):
    for invalid in ("97", "HH", "ZZ", ""):
        check = reference.copy(); check[column] = invalid
        assert HEADERS[column] + "_invalid" in normalize_record(check)[1]
for kind in NOISE_TYPES:
    check = reference.copy(); inject_noise(check, kind, random.Random(20))
    assert check[0] == reference[0] and check[1] in PRODUCTS and isinstance(check[2], float)
    assert check[6:10] == reference[6:10]
check = reference.copy()
selected, _ = inject_missing(check, random.Random(20))
assert all(HEADERS.index(name) not in PROTECTED_COLUMNS for name in selected)
assert check[0] and check[1] in PRODUCTS and isinstance(check[2], float) and all(check[i] for i in (6, 7, 8, 9))
assert sum(is_fintech_row(i) for i in range(1, 1001)) == 250
print("Passed: protected unique IDs, float amounts, three products, 25% one-fintech-leg design, and eligible-field noise.")


In [ ]:
started = time.perf_counter()
summary = Counter()
issue_counts, normalization_counts = Counter(), Counter()
blank_counts = Counter({header: 0 for header in HEADERS})
country_observed = {HEADERS[i]: Counter({"97": 0, "HH": 0, "ZZ": 0, "": 0}) for i in (10, 11)}
minimum_date = maximum_date = None
fintech_rows = 0
amount_parse_failures = 0
with tempfile.TemporaryDirectory() as temporary_dir:
    connection = sqlite3.connect(str(Path(temporary_dir) / "seen.sqlite"))
    connection.execute("CREATE TABLE seen (id TEXT PRIMARY KEY) WITHOUT ROWID")
    streams = []
    try:
        clean_writer = quarantine_writer = None
        if WRITE_CLEAN_OUTPUTS:
            for path in (CLEAN_FILE, QUARANTINE_FILE):
                streams.append(path.open("w", encoding="utf-8-sig", newline=""))
            clean_writer, quarantine_writer = [csv.writer(stream) for stream in streams]
            clean_writer.writerow(HEADERS)
            quarantine_writer.writerow(HEADERS)
        with OUTPUT_FILE.open(encoding="utf-8-sig", newline="") as handle:
            reader = csv.reader(handle, strict=True)
            if next(reader, None) != HEADERS: raise ValueError("Header mismatch")
            for line, original in enumerate(reader, 2):
                row, errors, fixes = normalize_record(original)
                if len(row) != len(HEADERS): raise ValueError(f"Malformed record at CSV line {line}")
                blank_counts.update(HEADERS[i] for i, value in enumerate(original) if value == "")
                if any(value == "" for value in original): summary["rows_with_any_blank"] += 1
                if any(original[i] == "" for i in range(len(HEADERS)) if i not in (6, 7)):
                    summary["rows_with_required_blank"] += 1
                for i in (10, 11):
                    if original[i] in ("97", "HH", "ZZ", ""):
                        country_observed[HEADERS[i]][original[i]] += 1
                if not row[0]: raise AssertionError("trxn_id must never be blank")
                cursor = connection.execute("INSERT OR IGNORE INTO seen VALUES (?)", (row[0],))
                if cursor.rowcount == 0: raise AssertionError("trxn_id must be unique")
                try:
                    parsed_amount = float(original[2])
                    if not (parsed_amount > 0 and parsed_amount < float("inf")): raise ValueError
                except (TypeError, ValueError, OverflowError):
                    amount_parse_failures += 1
                legs = int(is_fintech_name(original[8])) + int(is_fintech_name(original[9]))
                if legs > 1: raise AssertionError("A transaction cannot have two fintech legs")
                fintech_rows += int(legs == 1)
                summary["total_rows"] += 1
                issue_counts.update(errors)
                normalization_counts.update(fixes)
                if fixes: summary["rows_with_normalizations"] += 1
                if errors:
                    summary["quarantined_rows"] += 1
                    if quarantine_writer: quarantine_writer.writerow(original)
                else:
                    summary["accepted_rows"] += 1
                    summary["accepted_after_normalization" if fixes else "accepted_unchanged"] += 1
                    minimum_date = row[3] if minimum_date is None else min(minimum_date, row[3])
                    maximum_date = row[3] if maximum_date is None else max(maximum_date, row[3])
                    if clean_writer: clean_writer.writerow(row)
        if summary["total_rows"] != ROW_COUNT: raise ValueError("Unexpected row count")
        if amount_parse_failures: raise AssertionError("Every amount must parse as a positive finite float")
        if blank_counts["trxn_id"] or blank_counts["amount"]: raise AssertionError("Protected fields cannot be blank")
        if fintech_rows != round(ROW_COUNT * FINTECH_ROW_RATE): raise AssertionError("Fintech row share mismatch")
        if summary["accepted_rows"] + summary["quarantined_rows"] != ROW_COUNT:
            raise ValueError("Row reconciliation failed")
        if summary["quarantined_rows"] > generation_counts["rows_selected_for_noise_or_missing"]:
            raise ValueError("Unexpected quality errors in unmodified records")
    finally:
        connection.close()
        for stream in streams: stream.close()

report = {
    "seed": SEED, "noise_probability": NOISE_RATE, "expected_rows": ROW_COUNT,
    "focal_client_accounts": FOCAL_ACCOUNT_MAPPING,
    "schema": HEADERS, "missing_row_probability": MISSING_ROW_RATE,
    "max_missing_fields": MAX_MISSING_FIELDS,
    "generation_counts": dict(generation_counts),
    "missing_injection_selected_fields": dict(missing_selected_fields),
    "newly_blank_fields_from_missing_injection": dict(newly_blank_fields),
    "observed_blank_cells_by_column": dict(blank_counts),
    "observed_country_noise": {name: dict(counts) for name, counts in country_observed.items()},
    "valid_date_start": str(START_DATE), "valid_date_end": str(END_DATE),
    "injected_noise_rows": sum(injected.values()), "injected_noise_counts": dict(injected),
    "products_before_noise": dict(product_counts), "fintech_rows": fintech_rows,
    "fintech_row_rate": fintech_rows / ROW_COUNT, "amount_storage": "CSV numeric text parsed as Python float",
    "summary": dict(summary),
    "quality_issue_counts": dict(issue_counts), "normalized_field_counts": dict(normalization_counts),
    "accepted_date_min": minimum_date, "accepted_date_max": maximum_date,
    "clean_outputs_written": WRITE_CLEAN_OUTPUTS,
    "notes": "Issue counts can overlap. Raw data is never silently repaired. No financial-crime labels are inferred."
}
REPORT_FILE.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))
print(f"Full quality pass completed in {time.perf_counter() - started:.1f}s")
print(f"Raw CSV: {OUTPUT_FILE}")
print(f"Quality report: {REPORT_FILE}")
if WRITE_CLEAN_OUTPUTS: print(f"Clean: {CLEAN_FILE}\nQuarantine: {QUARANTINE_FILE}")


## Interpreting results

The report separates rows selected for noise or missing-value injection from rows actually rejected. It reports observed blanks for every original column and observed `97`, `HH`, `ZZ`, and blank values in both country columns. A selected field may already be blank; both selected-field counts and newly-created blank counts are included. The two independent injection probabilities can overlap and must not simply be added. Mixed-format dates and lowercase codes may be repairable. Invalid dates and missing identifiers are not silently imputed. Amount and transaction-ID errors are not injected. Error counts can exceed rejected rows because one row can fail several checks.

To produce accepted and rejected files, set WRITE_CLEAN_OUTPUTS=True and rerun. The accepted CSV retains the exact 12-column schema. The quarantine CSV preserves the same original 12 columns and the original values. Aggregate rejection reasons are stored separately in the JSON quality report, not as added dataset columns. These optional files require additional disk space; only use them after the full quality cell completes. The raw CSV remains unchanged. These controls address data quality, not AML detection or regulatory reporting requirements.


# Extended AFC Dataset: 291 Screenshot Fields

This section writes the original 291 screenshot fields as one header row and one transaction per row. It can run independently of the earlier generator when client_registry.py is available in the project folder. Both sections use the same client and account ownership definitions; transaction samples are independent.

The default is 1,000,000 rows. Valid dates range from 2025-01-01 to 2026-09-30. Products are Wires, Domestic ACH, and International ACH only. Exactly 25% of rows have one fintech core leg. Non-fintech focal businesses and ordinary individuals/businesses populate the remaining core legs.

WIDE_NOISE_RATE=0.15 selects exactly 15% of transaction rows for noise, not 15% of each column. Eligible fields include dates, cities, countries, address references, currency, classification codes, and optional free text. All names, account numbers, and entity IDs in all populated party groups are protected. Optional groups may be absent as a whole; their baseline blanks do not count as injected noise.

No columns are added. Existing output files are not updated until you run this section. The file and its full read-back audit can take several minutes. CSV accounts should be imported as text.


In [ ]:
# Configuration and explicit schema transcription from screenshot column A.
from pathlib import Path
from datetime import date, timedelta
from collections import Counter
import csv
import json
import os
import random
import re
import sqlite3
import tempfile
import time

WIDE_ROW_COUNT = 1_000_000
WIDE_NOISE_RATE = 0.15  # Fraction of transaction rows; allowed range: 0.10 to 0.20.
WIDE_SEED = 20261001
WIDE_OVERWRITE = True
WIDE_FINTECH_ROW_RATE = 0.25
WIDE_PROJECT_DIR = None  # Example: Path(r"C:\FCO-Review-main")

WIDE_TRANSACTION_FIELDS = [
    "trxn_id", "amount", "fc_amount", "fc_type", "run_date", "value_date", "load_date",
    "lvl_1_product_code", "lvl_1_product_text", "lvl_2_product_code", "lvl_2_product_txt",
    "lvl_3_product_text", "src_sys_code", "src_table_id", "is_202_cov", "instr_type",
    "instr_nm", "instr_routg_cd", "aug_srl_nb",
]
WIDE_PARTIES = [
    "orig", "obk", "d_int_1", "d_int_2", "sbk", "rbk", "c_int_5", "c_int_6", "bbk", "bene",
    "mi_rem", "mi_iss_instn", "mi_clr_instn", "mi_dep_instn", "mi_bene", "mi_scnd_bene",
    "ult_orig", "ult_bene", "ip",
]
WIDE_PARTY_FIELDS = [
    "entity_id", "lob_code", "type", "raw_account_num", "std_account_num", "raw_name",
    "clean_name", "master_name", "raw_addr_id", "cntry_code", "city", "segment_id",
    "is_in_accounts_master", "is_bank",
]
WIDE_TRAILING_FIELDS = ["instruction", "text_msg_1a", "lob_code", "load_year", "load_month", "lvl_3_product_code"]
WIDE_HEADERS = (WIDE_TRANSACTION_FIELDS +
                [f"{party}_{field}" for party in WIDE_PARTIES for field in WIDE_PARTY_FIELDS] +
                WIDE_TRAILING_FIELDS)
assert len(WIDE_HEADERS) == len(set(WIDE_HEADERS)) == 291
# Screenshot row-number anchors verify group boundaries and spelling/order.
for screenshot_row, field in {
    1: "trxn_id", 19: "aug_srl_nb", 20: "orig_entity_id", 34: "obk_entity_id",
    48: "d_int_1_entity_id", 62: "d_int_2_entity_id", 76: "sbk_entity_id", 90: "rbk_entity_id",
    104: "c_int_5_entity_id", 118: "c_int_6_entity_id", 132: "bbk_entity_id", 146: "bene_entity_id",
    160: "mi_rem_entity_id", 174: "mi_iss_instn_entity_id", 188: "mi_clr_instn_entity_id",
    202: "mi_dep_instn_entity_id", 216: "mi_bene_entity_id", 230: "mi_scnd_bene_entity_id",
    244: "ult_orig_entity_id", 258: "ult_bene_entity_id", 272: "ip_entity_id",
    286: "instruction", 291: "lvl_3_product_code",
}.items():
    assert WIDE_HEADERS[screenshot_row - 1] == field

if not isinstance(WIDE_ROW_COUNT, int) or isinstance(WIDE_ROW_COUNT, bool) or not 1 <= WIDE_ROW_COUNT <= 9_999_999:
    raise ValueError("WIDE_ROW_COUNT must be between 1 and 9,999,999.")
if not 0.10 <= WIDE_NOISE_RATE <= 0.20:
    raise ValueError("WIDE_NOISE_RATE must be between 0.10 and 0.20.")
if WIDE_PROJECT_DIR is None:
    WIDE_PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents]
                             if (p / "convert.py").is_file() and (p / "mapping.py").is_file()), None)
if WIDE_PROJECT_DIR is None:
    raise RuntimeError("Set WIDE_PROJECT_DIR to the project folder.")
WIDE_PROJECT_DIR = Path(WIDE_PROJECT_DIR).expanduser().resolve()
if not WIDE_PROJECT_DIR.is_dir(): raise ValueError("WIDE_PROJECT_DIR must exist.")
WIDE_OUTPUT_DIR = WIDE_PROJECT_DIR / "output"
WIDE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WIDE_OUTPUT_FILE = WIDE_OUTPUT_DIR / "synthetic_afc_extended_1m.csv"
WIDE_REPORT_FILE = WIDE_OUTPUT_DIR / "synthetic_afc_extended_quality_report.json"
WIDE_INDEX = {name: i for i, name in enumerate(WIDE_HEADERS)}
WIDE_GROUP_STARTS = {party: WIDE_INDEX[party + "_entity_id"] for party in WIDE_PARTIES}
WIDE_START_DATE, WIDE_END_DATE = date(2025, 1, 1), date(2026, 9, 30)
WIDE_DATES = [(WIDE_START_DATE + timedelta(days=i)).strftime("%Y%m%d")
              for i in range((WIDE_END_DATE - WIDE_START_DATE).days + 1)]
WIDE_DATE_SET = set(WIDE_DATES)
print(f"Extended schema: {len(WIDE_HEADERS)} fields; rows: {WIDE_ROW_COUNT:,}; noise: {WIDE_NOISE_RATE:.0%}")
print(f"CSV: {WIDE_OUTPUT_FILE}")
print("First 19 headers:", WIDE_TRANSACTION_FIELDS)
print("Party group order:", WIDE_PARTIES)
print("Last six headers:", WIDE_TRAILING_FIELDS)


## Stable parties, accounts, and related roles

The registry defines each client's canonical name, stable entity ID, country/city, and 3, 5, or 7 accounts. Raw and standard account columns use the same fixed account token; raw, clean, and master names use the same canonical client name. No client names contain synthetic prefixes, test-account wording, or numeric transaction suffixes. Accounts are exclusive to their owner and reusable across both datasets and both core legs.

When populated, ult_orig reuses the orig client and selected account, and ult_bene reuses bene. The initiating party reuses a focal core leg when available (beneficiary if focal, otherwise originator). Banks and intermediary roles select their own registered clients and accounts. Optional roles retain the configured presence probabilities.

Baseline value_date <= run_date <= load_date, and load_year/month follow load_date. Account-owner country determines Domestic ACH and International ACH counterparty selection. Country/city noise can deliberately break location consistency without changing account ownership. Both amount fields remain positive floats with the existing illustrative FX relationship.


In [ ]:
# Tuple order: level-1 code/text, level-2 code/text, level-3 code/text, instruction type.
WIDE_PRODUCTS = [
    ("1", "Wires", "101", "Wires", "10101", "Wires", "103"),
    ("2", "Domestic ACH", "201", "Domestic ACH", "20101", "Domestic ACH", "201"),
    ("2", "International ACH", "202", "International ACH", "20201", "International ACH", "202"),
]
WIDE_PRODUCT_WEIGHTS = (40, 35, 25)
# Test-only USD-per-currency factors represented as integer hundredths.
WIDE_FX_FACTORS = {"USD": 100, "EUR": 110, "GBP": 125, "CAD": 75, "AUD": 65, "SGD": 74}
import sys
if str(WIDE_PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(WIDE_PROJECT_DIR))
from client_registry import COUNTRY_CITIES, FINTECH_NAMES, build_registry, choose_pair, party_fields

WIDE_COUNTRY_CITIES = COUNTRY_CITIES
WIDE_COUNTRIES = tuple(COUNTRY_CITIES)
WIDE_ALL_CITIES = {city for cities in COUNTRY_CITIES.values() for city in cities}
WIDE_CLIENTS, WIDE_ACCOUNT_OWNERS, WIDE_CLIENT_POOLS = build_registry()
WIDE_FINTECH_COMPANIES = tuple(name.upper() for name in FINTECH_NAMES)
WIDE_FOCAL_ACCOUNT_MAPPING = {name: list(client["accounts"])
                              for name, client in WIDE_CLIENTS.items() if client["focal"]}

WIDE_ROLE_PROBABILITY = {
    "orig": 1.0, "bene": 1.0, "obk": 0.88, "bbk": 0.88, "sbk": 0.80, "rbk": 0.80,
    "d_int_1": 0.18, "d_int_2": 0.08, "c_int_5": 0.12, "c_int_6": 0.06,
    "mi_rem": 0.06, "mi_iss_instn": 0.04, "mi_clr_instn": 0.04, "mi_dep_instn": 0.04,
    "mi_bene": 0.06, "mi_scnd_bene": 0.03, "ult_orig": 0.25, "ult_bene": 0.25, "ip": 0.35,
}
WIDE_BANK_ROLES = {"obk", "bbk", "sbk", "rbk", "d_int_1", "d_int_2", "c_int_5", "c_int_6",
                   "mi_iss_instn", "mi_clr_instn", "mi_dep_instn"}

def wide_clean_record(rng, sequence):
    row = [""] * len(WIDE_HEADERS)
    product = rng.choices(WIDE_PRODUCTS, weights=WIDE_PRODUCT_WEIGHTS, k=1)[0]
    product_name = product[3]
    day_index = rng.randrange(len(WIDE_DATES))
    run_index = min(day_index + rng.randrange(3), len(WIDE_DATES) - 1)
    load_index = min(run_index + rng.randrange(3), len(WIDE_DATES) - 1)
    value_day, run_day, load_day = WIDE_DATES[day_index], WIDE_DATES[run_index], WIDE_DATES[load_index]
    currency = rng.choice(tuple(WIDE_FX_FACTORS))
    foreign_amount = round(rng.uniform(1.00, 25_000_000.00), 2)
    amount = round(foreign_amount * WIDE_FX_FACTORS[currency] / 100, 2)
    row[:19] = [f"{value_day}{rng.choice(('C', 'S'))}{sequence:07d}", amount, foreign_amount,
                currency, run_day, value_day, load_day, product[0], product[1], product[2], product[3],
                product[5], rng.choice(("SYN_CORE", "SYN_WALLET", "SYN_GATEWAY")),
                rng.choice(("1", "2", "3")), rng.choice(("0", "1")), product[6],
                f"SYNTHETIC {product_name.upper()}", f"{rng.randrange(1000000000):09d}", str(sequence)]
    orig, bene = choose_pair(rng, sequence, product_name, WIDE_CLIENTS, WIDE_CLIENT_POOLS)
    orig_account, bene_account = rng.choice(orig["accounts"]), rng.choice(bene["accounts"])
    for party in WIDE_PARTIES:
        if rng.random() >= WIDE_ROLE_PROBABILITY[party]:
            continue
        if party in ("orig", "ult_orig"):
            client, account = orig, orig_account
        elif party in ("bene", "ult_bene"):
            client, account = bene, bene_account
        elif party == "ip":
            # The initiating party reuses the focal leg when present, including its account.
            client, account = (bene, bene_account) if bene["focal"] else (orig, orig_account)
        else:
            if party in ("obk", "sbk"):
                country = orig["country"]
            elif party in ("bbk", "rbk"):
                country = bene["country"]
            else:
                country = rng.choice(WIDE_COUNTRIES)
            kind = "bank" if party in WIDE_BANK_ROLES else "business"
            client = rng.choice(WIDE_CLIENT_POOLS[country, kind])
            account = rng.choice(client["accounts"])
        start = WIDE_GROUP_STARTS[party]
        row[start:start + 14] = party_fields(client, account)
    row[-6:] = [rng.choice(("", "SYNTHETIC PAYOUT", "SYNTHETIC WALLET FUNDING", "SYNTHETIC INVOICE PAYMENT")),
                "" if rng.random() < 0.6 else f"SYNTHETIC ORDER {sequence:09d}", "1",
                load_day[:4], str(int(load_day[4:6])), product[4]]
    return row


## Noise rules with protected ownership

Protected fields include transaction ID, amounts, product hierarchy, and every party's entity_id, raw_account_num, std_account_num, raw_name, clean_name, and master_name. No partial identity blanks or account/entity corruption are injected.

| Eligible field family | Noise |
|---|---|
| Cities | NOT FOUND, STREET, blank |
| Countries | 97, HH, ZZ, lowercase, blank |
| Dates | 99999999, 20260230, YYYY/MM/DD, blank |
| Currency | XXX, blank |
| Address references | BAD_ADDR_ID, blank |
| Flags | X, blank |
| Type, segment, party LOB codes | 97, blank |
| Load month | 13, blank |
| Routing code | ROUTE???, blank |
| Instruction | NOT PROVIDED, blank |
| Other unprotected fields | Blank |

Only populated, unprotected fields are eligible for the multiple-blanks scenario. Entire optional party groups can be naturally absent. Ownership always remains recoverable from either account leg.


In [ ]:
WIDE_CITY_COLUMNS = [WIDE_INDEX[p + "_city"] for p in WIDE_PARTIES]
WIDE_COUNTRY_COLUMNS = [WIDE_INDEX[p + "_cntry_code"] for p in WIDE_PARTIES]
WIDE_NOISE_KINDS = (
    "city_not_found", "city_street", "country_97", "country_hh", "country_zz", "country_lowercase",
    "date_sentinel", "date_impossible", "date_format", "date_missing", "currency_invalid", "address_invalid", "flag_invalid", "classification_invalid",
    "load_month_invalid", "routing_invalid", "multiple_blanks", "free_text_placeholder",
)

def wide_populated_column(row, rng, suffix):
    candidates = [WIDE_INDEX[p + "_" + suffix] for p in WIDE_PARTIES
                  if row[WIDE_INDEX[p + "_" + suffix]]]
    return rng.choice(candidates) if candidates else None

# Protect identity for every populated role, including banks and intermediaries.
WIDE_PROTECTED_COLUMNS = {0, 1, 2, 7, 8, 9, 10, 11, 15, 290}
WIDE_PROTECTED_COLUMNS.update(
    WIDE_INDEX[party + "_" + field]
    for party in WIDE_PARTIES
    for field in ("entity_id", "raw_account_num", "std_account_num", "raw_name", "clean_name", "master_name")
)

def wide_inject_noise(row, rng, kind):
    changed = []
    def put(column, value):
        if row[column] != value:
            row[column] = value
            changed.append(column)
    if kind in ("city_not_found", "city_street"):
        put(wide_populated_column(row, rng, "city"), "NOT FOUND" if kind == "city_not_found" else "STREET")
    elif kind.startswith("country_"):
        column = wide_populated_column(row, rng, "cntry_code")
        value = {"country_97": "97", "country_hh": "HH", "country_zz": "ZZ"}.get(kind, row[column].lower())
        put(column, value)
    elif kind.startswith("date_"):
        column = rng.choice([WIDE_INDEX[field] for field in ("run_date", "value_date", "load_date")])
        current = row[column]
        value = {"date_sentinel": "99999999", "date_impossible": "20260230", "date_missing": ""}.get(kind)
        if kind == "date_format": value = f"{current[:4]}/{current[4:6]}/{current[6:]}"
        put(column, value)
    elif kind == "currency_invalid": put(3, "XXX")
    elif kind == "address_invalid":
        column = wide_populated_column(row, rng, "raw_addr_id")
        if column is None: column = WIDE_INDEX["orig_raw_addr_id"]
        put(column, "BAD_ADDR_ID")
    elif kind == "flag_invalid":
        put(wide_populated_column(row, rng, rng.choice(("is_bank", "is_in_accounts_master"))), "X")
    elif kind == "classification_invalid":
        put(wide_populated_column(row, rng, rng.choice(("type", "segment_id", "lob_code"))), "97")
    elif kind == "load_month_invalid": put(WIDE_INDEX["load_month"], "13")
    elif kind == "routing_invalid": put(WIDE_INDEX["instr_routg_cd"], "ROUTE???")
    elif kind == "multiple_blanks":
        candidates = [i for i, value in enumerate(row) if value and i not in WIDE_PROTECTED_COLUMNS]
        for column in rng.sample(candidates, min(rng.randint(1, 3), len(candidates))):
            put(column, "")
    elif kind == "free_text_placeholder": put(WIDE_INDEX["instruction"], "NOT PROVIDED")
    else: raise ValueError(f"Unknown noise kind: {kind}")
    if not changed: raise AssertionError(f"Noise scenario did not change the row: {kind}")
    return changed


## Quality rules

The quality functions retain existing date, product, amount, location, and code checks. Each populated party group is additionally checked against WIDE_ACCOUNT_OWNERS: account ownership, stable entity ID, raw/standard account agreement, and all three names must match the same registered client. Names are compared with the configured canonical spelling rather than requiring a synthetic prefix.

The existing preflight checks below are retained for future notebook execution. No cells or tests have been run as part of this code-only revision.


In [ ]:
WIDE_ID_RE = re.compile(r"[0-9]{8}[CS][0-9]{7}")
WIDE_ENTITY_RE = re.compile(r"[0-9a-f]{32}")
WIDE_ACCOUNT_RE = re.compile(r"[0-9]{14}")
WIDE_ADDRESS_RE = re.compile(r"SYNADDR[0-9]{12}")
WIDE_PRODUCT_LOOKUP = {p[3].casefold(): p for p in WIDE_PRODUCTS}
WIDE_REQUIRED_CORE = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 15, 287, 288, 289, 290]
WIDE_DATE_COLUMNS = (4, 5, 6)

def wide_check_row(original):
    if len(original) != len(WIDE_HEADERS): raise ValueError("Expected exactly 291 fields")
    row = [value.strip() if isinstance(value, str) else value for value in original]
    errors = []
    for i in WIDE_REQUIRED_CORE:
        if not row[i]: errors.append(WIDE_HEADERS[i] + ":missing")
    for i in WIDE_DATE_COLUMNS:
        value = row[i]
        if len(value) == 10 and value[4] == value[7] and value[4] in ("/", "-"):
            value = value[:4] + value[5:7] + value[8:]
        if value not in WIDE_DATE_SET: errors.append(WIDE_HEADERS[i] + ":invalid_date")
        else: row[i] = value
    key = row[0]
    if not WIDE_ID_RE.fullmatch(key) or key[:8] not in WIDE_DATE_SET: errors.append("trxn_id:invalid")
    elif row[5] in WIDE_DATE_SET and key[:8] != row[5]: errors.append("trxn_id:value_date_mismatch")
    if all(row[i] in WIDE_DATE_SET for i in WIDE_DATE_COLUMNS):
        if not row[5] <= row[4] <= row[6]: errors.append("dates:chronology")
    for i in (1, 2):
        try:
            value = float(row[i])
            if not (value > 0 and value < float("inf")): raise ValueError
            row[i] = f"{value:.2f}"
        except (TypeError, ValueError, OverflowError):
            errors.append(WIDE_HEADERS[i] + ":invalid_amount")
    row[3] = row[3].upper()
    if row[3] not in WIDE_FX_FACTORS: errors.append("fc_type:invalid_currency")
    else:
        try:
            if abs(float(row[1]) - round(float(row[2]) * WIDE_FX_FACTORS[row[3]] / 100, 2)) > 0.001:
                errors.append("amount:fx_relationship")
        except (TypeError, ValueError):
            pass
    product = WIDE_PRODUCT_LOOKUP.get(" ".join(row[10].split()).casefold())
    if product is None: errors.append("lvl_2_product_txt:invalid")
    else:
        row[10] = product[3]
        for i, expected in zip((7, 8, 9, 11, 290, 15), (product[0], product[1], product[2], product[5], product[4], product[6])):
            if row[i] != expected: errors.append(WIDE_HEADERS[i] + ":product_hierarchy")
    if row[12] and row[12] not in ("SYN_CORE", "SYN_WALLET", "SYN_GATEWAY"): errors.append("src_sys_code:invalid")
    if row[13] and row[13] not in ("1", "2", "3"): errors.append("src_table_id:invalid")
    if row[14] and row[14] not in ("0", "1"): errors.append("is_202_cov:invalid")
    if row[17] and (not row[17].isascii() or not row[17].isdigit() or len(row[17]) != 9):
        errors.append("instr_routg_cd:invalid")
    if row[18] and (not row[18].isascii() or not row[18].isdigit()): errors.append("aug_srl_nb:invalid")
    for party, start in WIDE_GROUP_STARTS.items():
        if not any(row[start:start + 14]) and party not in ("orig", "bene"): continue
        if party in ("orig", "bene"):
            for offset in (0, 3, 6, 9, 10):
                if not row[start + offset]: errors.append(WIDE_HEADERS[start + offset] + ":missing")
        for offset, allowed in ((1, {"1", "2", "3"}), (2, {"1", "2"}), (11, {"0", "1", "2", "3"}),
                                (12, {"0", "1"}), (13, {"0", "1"})):
            if row[start + offset] and row[start + offset] not in allowed:
                errors.append(WIDE_HEADERS[start + offset] + ":invalid_code")
        for offset, pattern in ((0, WIDE_ENTITY_RE), (3, WIDE_ACCOUNT_RE), (4, WIDE_ACCOUNT_RE), (8, WIDE_ADDRESS_RE)):
            if row[start + offset] and not pattern.fullmatch(row[start + offset]):
                errors.append(WIDE_HEADERS[start + offset] + ":invalid_identifier")
        owner = WIDE_ACCOUNT_OWNERS.get(row[start + 3])
        if owner is None:
            errors.append(WIDE_HEADERS[start + 3] + ":unknown_account")
        else:
            if row[start] != owner["entity_id"]:
                errors.append(WIDE_HEADERS[start] + ":ownership_mismatch")
            if row[start + 4] != row[start + 3]:
                errors.append(WIDE_HEADERS[start + 4] + ":account_mismatch")
            for offset in (5, 6, 7):
                if row[start + offset] != owner["name"]:
                    errors.append(WIDE_HEADERS[start + offset] + ":ownership_mismatch")
        country, city = start + 9, start + 10
        row[country] = row[country].upper()
        row[city] = " ".join(row[city].split()).upper()
        if row[country] and row[country] not in WIDE_COUNTRY_CITIES: errors.append(WIDE_HEADERS[country] + ":invalid_country")
        if row[city] and row[city] not in WIDE_ALL_CITIES: errors.append(WIDE_HEADERS[city] + ":invalid_city")
        elif row[city] and row[country] in WIDE_COUNTRY_CITIES and row[city] not in WIDE_COUNTRY_CITIES[row[country]]:
            errors.append(WIDE_HEADERS[city] + ":country_mismatch")
    if row[6] in WIDE_DATE_SET:
        if row[288] != row[6][:4]: errors.append("load_year:date_mismatch")
        if row[289] != str(int(row[6][4:6])): errors.append("load_month:date_mismatch")
    if row[287] not in ("1", "2", "3"): errors.append("lob_code:invalid")
    for i in (16, 285, 286):
        if row[i] and not row[i].startswith("SYNTHETIC "): errors.append(WIDE_HEADERS[i] + ":placeholder")
    oc, bc = row[WIDE_INDEX["orig_cntry_code"]], row[WIDE_INDEX["bene_cntry_code"]]
    if oc in WIDE_COUNTRY_CITIES and bc in WIDE_COUNTRY_CITIES:
        if row[10] == "International ACH" and oc == bc: errors.append("corridor:expected_cross_border")
        if row[10] == "Domestic ACH" and oc != bc: errors.append("corridor:expected_domestic")
    fixes = [WIDE_HEADERS[i] for i, (before, after) in enumerate(zip(original, row)) if before != after]
    return row, errors, fixes

# Test every scenario and every populated city/country role before generating the large file.
wide_test_rng = random.Random(WIDE_SEED + 20)
for wide_test_sequence in range(1, 301):
    wide_reference = wide_clean_record(wide_test_rng, wide_test_sequence)
    assert len(wide_reference) == 291
    assert not wide_check_row(wide_reference)[1]
for wide_kind in WIDE_NOISE_KINDS:
    wide_mutated = wide_reference.copy()
    wide_changes = wide_inject_noise(wide_mutated, random.Random(31), wide_kind)
    assert wide_changes and len(wide_mutated) == 291
    _, wide_errors, wide_fixes = wide_check_row(wide_mutated)
    if wide_kind not in ("multiple_blanks",): assert wide_errors or wide_fixes
    assert wide_mutated[0] == wide_reference[0]
    assert isinstance(wide_mutated[1], float) and isinstance(wide_mutated[2], float)
    assert wide_mutated[10] in {p[3] for p in WIDE_PRODUCTS}
    assert all(wide_mutated[i] == wide_reference[i] for i in WIDE_PROTECTED_COLUMNS)
for wide_party, wide_start in WIDE_GROUP_STARTS.items():
    for wide_offset, wide_token in ((9, "97"), (9, "HH"), (10, "NOT FOUND"), (10, "STREET")):
        wide_mutated = wide_reference.copy()
        wide_mutated[wide_start + wide_offset] = wide_token
        assert any(WIDE_HEADERS[wide_start + wide_offset] in error for error in wide_check_row(wide_mutated)[1])
assert sum((i - 1) % 4 == 0 for i in range(1, 1001)) == 250
print("Passed: protected unique IDs, float amounts, three products, 25% one-fintech-leg design, and all city/country groups.")


## Stream the million-row CSV

An exact-count random selection algorithm chooses noisy rows without allocating a million-row matrix or a large set of row numbers. The probability at each step is remaining noisy rows / remaining rows. The generator uses separate seeded streams for clean data, noise selection, and noise details, and writes through a temporary file before replacing the final CSV. Repeated runs with the same settings are deterministic.


In [ ]:
if not WIDE_OVERWRITE and any(p.exists() for p in (WIDE_OUTPUT_FILE, WIDE_REPORT_FILE)):
    raise FileExistsError("Extended output exists. Set WIDE_OVERWRITE=True to replace it.")
wide_rng = random.Random(WIDE_SEED)
wide_selection_rng = random.Random(WIDE_SEED + 1)
wide_noise_rng = random.Random(WIDE_SEED + 2)
wide_noise_target = round(WIDE_ROW_COUNT * WIDE_NOISE_RATE)
wide_noise_remaining = wide_noise_target
wide_noise_counts = Counter()
wide_changed_columns = Counter()
wide_product_counts = Counter()
wide_started = time.perf_counter()
wide_descriptor, wide_temp_name = tempfile.mkstemp(dir=WIDE_OUTPUT_DIR, suffix=".csv.tmp")
os.close(wide_descriptor)
wide_temp_path = Path(wide_temp_name)
try:
    with wide_temp_path.open("w", encoding="utf-8-sig", newline="") as wide_handle:
        wide_writer = csv.writer(wide_handle)
        wide_writer.writerow(WIDE_HEADERS)
        for wide_sequence in range(1, WIDE_ROW_COUNT + 1):
            wide_row = wide_clean_record(wide_rng, wide_sequence)
            wide_product_counts[wide_row[10]] += 1
            if wide_selection_rng.randrange(WIDE_ROW_COUNT - wide_sequence + 1) < wide_noise_remaining:
                wide_kind = wide_noise_rng.choice(WIDE_NOISE_KINDS)
                wide_changed = wide_inject_noise(wide_row, wide_noise_rng, wide_kind)
                wide_noise_counts[wide_kind] += 1
                wide_changed_columns.update(WIDE_HEADERS[i] for i in wide_changed)
                wide_noise_remaining -= 1
            wide_writer.writerow(wide_row)
            if wide_sequence % 100_000 == 0:
                print(f"Extended rows written: {wide_sequence:,}; elapsed: {time.perf_counter() - wide_started:.1f}s", flush=True)
    assert wide_noise_remaining == 0 and sum(wide_noise_counts.values()) == wide_noise_target
    os.replace(wide_temp_path, WIDE_OUTPUT_FILE)
finally:
    wide_temp_path.unlink(missing_ok=True)
print(f"Generated {WIDE_ROW_COUNT:,} rows x {len(WIDE_HEADERS)} columns; noisy rows: {wide_noise_target:,}")
print(f"Size: {WIDE_OUTPUT_FILE.stat().st_size / 1024**3:.2f} GiB; elapsed: {time.perf_counter() - wide_started:.1f}s")


## Full saved-file audit

Read the exported CSV again to validate header order and field count for every row, normalize only unambiguous formats in memory, prove every transaction ID is present and unique, and reconcile all accepted/flagged records. Report observed city/country tokens and blanks per column. Baseline absence is not the same as injected noise. The raw CSV keeps all records and is never rewritten by this audit. The JSON report holds the complete field list and detailed counts outside the dataset.

Flagged rows need investigation under this test contract. Accepted rows may include legitimate blanks in optional intermediary groups. A low rejection count does not mean a low noise injection rate because some noise is safely normalizable and some missing fields are optional. No clean/quarantine CSVs are written by this section; extend the streaming writer only if those additional artifacts are needed.


In [ ]:
wide_audit_started = time.perf_counter()
wide_summary = Counter()
wide_issues = Counter()
wide_normalizations = Counter()
wide_blank_counts = [0] * len(WIDE_HEADERS)
wide_dirty_tokens = {WIDE_HEADERS[i]: Counter() for i in WIDE_CITY_COLUMNS + WIDE_COUNTRY_COLUMNS}
wide_date_bounds = {field: [None, None] for field in ("run_date", "value_date", "load_date")}
wide_numeric_columns = {WIDE_INDEX[f] for f in ("amount", "fc_amount")}
wide_fintech_rows = 0
wide_amount_parse_failures = 0
with tempfile.TemporaryDirectory() as wide_temporary_dir:
    wide_connection = sqlite3.connect(str(Path(wide_temporary_dir) / "wide_seen.sqlite"))
    wide_connection.execute("CREATE TABLE seen (id TEXT PRIMARY KEY) WITHOUT ROWID")
    try:
        with WIDE_OUTPUT_FILE.open(encoding="utf-8-sig", newline="") as wide_handle:
            wide_reader = csv.reader(wide_handle, strict=True)
            if next(wide_reader, None) != WIDE_HEADERS: raise ValueError("Extended header mismatch")
            for wide_count, wide_original in enumerate(wide_reader, 1):
                if len(wide_original) != 291: raise ValueError(f"Wrong field count in record {wide_count}")
                wide_normalized, wide_errors, wide_fixes = wide_check_row(wide_original)
                wide_key = wide_normalized[0]
                if not wide_key: raise AssertionError("trxn_id must never be blank")
                wide_cursor = wide_connection.execute("INSERT OR IGNORE INTO seen VALUES (?)", (wide_key,))
                if wide_cursor.rowcount == 0: raise AssertionError("trxn_id must be unique")
                try:
                    if not all(float(wide_original[i]) > 0 for i in wide_numeric_columns): raise ValueError
                except (TypeError, ValueError, OverflowError):
                    wide_amount_parse_failures += 1
                orig_fintech = any(company in wide_original[WIDE_INDEX["orig_clean_name"]].upper() for company in WIDE_FINTECH_COMPANIES)
                bene_fintech = any(company in wide_original[WIDE_INDEX["bene_clean_name"]].upper() for company in WIDE_FINTECH_COMPANIES)
                if orig_fintech and bene_fintech: raise AssertionError("Both core legs cannot be fintech")
                wide_fintech_rows += int(orig_fintech or bene_fintech)
                for i, value in enumerate(wide_original):
                    if value == "": wide_blank_counts[i] += 1
                for i in WIDE_CITY_COLUMNS:
                    if wide_original[i] in ("NOT FOUND", "STREET"):
                        wide_dirty_tokens[WIDE_HEADERS[i]][wide_original[i]] += 1
                for i in WIDE_COUNTRY_COLUMNS:
                    if wide_original[i] in ("97", "HH", "ZZ"):
                        wide_dirty_tokens[WIDE_HEADERS[i]][wide_original[i]] += 1
                wide_issues.update(wide_errors)
                wide_normalizations.update(wide_fixes)
                wide_summary["total_rows"] += 1
                if wide_errors: wide_summary["flagged_rows"] += 1
                else:
                    wide_summary["accepted_rows"] += 1
                    wide_summary["accepted_after_normalization" if wide_fixes else "accepted_unchanged"] += 1
                    for field, bounds in wide_date_bounds.items():
                        value = wide_normalized[WIDE_INDEX[field]]
                        bounds[0] = value if bounds[0] is None else min(bounds[0], value)
                        bounds[1] = value if bounds[1] is None else max(bounds[1], value)
                if wide_fixes: wide_summary["rows_with_normalizations"] += 1
                if wide_count % 100_000 == 0:
                    print(f"Extended rows audited: {wide_count:,}; elapsed: {time.perf_counter() - wide_audit_started:.1f}s", flush=True)
    finally:
        wide_connection.close()
assert wide_summary["total_rows"] == WIDE_ROW_COUNT
assert not wide_amount_parse_failures
assert wide_blank_counts[0] == wide_blank_counts[1] == wide_blank_counts[2] == 0
assert wide_fintech_rows == round(WIDE_ROW_COUNT * WIDE_FINTECH_ROW_RATE)
assert wide_summary["accepted_rows"] + wide_summary["flagged_rows"] == WIDE_ROW_COUNT
assert wide_summary["flagged_rows"] <= wide_noise_target
wide_report = {
    "focal_client_accounts": WIDE_FOCAL_ACCOUNT_MAPPING,
    "schema": WIDE_HEADERS, "field_count": len(WIDE_HEADERS), "row_count": WIDE_ROW_COUNT,
    "orientation": "one header row, one transaction per subsequent row",
    "seed": WIDE_SEED, "noise_unit": "transaction rows", "noise_rate": WIDE_NOISE_RATE,
    "exact_noisy_rows": wide_noise_target, "noise_scenarios": dict(wide_noise_counts),
    "changed_columns": dict(wide_changed_columns), "baseline_product_counts": dict(wide_product_counts),
    "fintech_rows": wide_fintech_rows, "fintech_row_rate": wide_fintech_rows / WIDE_ROW_COUNT,
    "amount_storage": "CSV numeric text parsed as Python float",
    "summary": dict(wide_summary), "quality_issue_counts": dict(wide_issues),
    "normalized_field_counts": dict(wide_normalizations),
    "blank_cells_by_column": dict(zip(WIDE_HEADERS, wide_blank_counts)),
    "city_country_noise_by_column": {name: dict(counts) for name, counts in wide_dirty_tokens.items()},
    "accepted_date_bounds": wide_date_bounds,
    "valid_date_start": str(WIDE_START_DATE), "valid_date_end": str(WIDE_END_DATE),
    "notes": "Natural optional blanks are not counted as injected noise. Issue counts overlap. No actual financial-crime labels or market FX rates are used.",
}
wide_report_temp = WIDE_REPORT_FILE.with_suffix(".json.tmp")
wide_report_temp.write_text(json.dumps(wide_report, indent=2), encoding="utf-8")
os.replace(wide_report_temp, WIDE_REPORT_FILE)
print(json.dumps({"summary": wide_report["summary"], "noise_rows": wide_noise_target,
                  "date_bounds": wide_date_bounds, "noise_scenarios": dict(wide_noise_counts)}, indent=2))
print(f"Audit complete in {time.perf_counter() - wide_audit_started:.1f}s")
print(f"Raw extended CSV: {WIDE_OUTPUT_FILE}")
print(f"Extended quality report: {WIDE_REPORT_FILE}")


# Distinct Values for Both Generated Datasets

Run this independent section after the two CSVs exist. It profiles **every column** in the 12-column dataset and the 291-column dataset. It does not alter either CSV or add fields to them.

Distinct values are compared as exact raw strings: case, leading zeros, and surrounding whitespace remain significant. An empty field (`""`) is included as one distinct value when present; its number of occurrences is reported separately. Strings such as `NULL`, `97`, `HH`, `NOT FOUND`, and `STREET` are literal values, not automatically converted to missing data.

For each input, two CSV reports are written into `output`:

1. `<input_stem>_distinct_summary.csv`: one row per original column, with distinct counts including/excluding blank, blank/nonblank row counts, total rows, and a JSON preview of up to eight distinct values.
2. `<input_stem>_distinct_values.csv`: the **complete** distinct-value list for every column, including the exact value, an explicit blank flag, and its occurrence count. High-cardinality columns are not truncated. These reports may be large.

The notebook displays counts and previews for all columns, while the full list remains in the values report. Both reports preserve the input column order; values within each column are sorted. All count totals are reconciled to the input row count. A value that looks numeric is still handled as text; import the report's value column as Text in Excel if needed.

The implementation reads each dataset once in chunks. Small dictionaries count low-cardinality columns; large dictionaries are flushed into a temporary SQLite database. This avoids retaining all million transaction IDs in Python memory. The temporary database is deleted after export. Reports use temporary files and replace final files only after profiling succeeds. The default is to replace existing distinct reports, not the source datasets.


In [ ]:
from pathlib import Path
import csv
import json
import os
import sqlite3
import tempfile
import time

DISTINCT_PROJECT_DIR = None  # Example: Path(r"C:\FCO-Review-main")
DISTINCT_CHUNK_ROWS = 10_000
DISTINCT_MEMORY_VALUES_PER_COLUMN = 5_000
DISTINCT_PREVIEW_VALUES = 8
DISTINCT_OVERWRITE = True

if DISTINCT_PROJECT_DIR is None:
    DISTINCT_PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents]
                                 if (p / "convert.py").is_file() and (p / "mapping.py").is_file()), None)
if DISTINCT_PROJECT_DIR is None: raise RuntimeError("Set DISTINCT_PROJECT_DIR to the project folder.")
DISTINCT_OUTPUT_DIR = Path(DISTINCT_PROJECT_DIR).expanduser().resolve() / "output"
DISTINCT_INPUTS = [DISTINCT_OUTPUT_DIR / "synthetic_wire_transactions_1m.csv",
                   DISTINCT_OUTPUT_DIR / "synthetic_afc_extended_1m.csv"]
if DISTINCT_CHUNK_ROWS <= 0 or DISTINCT_MEMORY_VALUES_PER_COLUMN <= 0 or DISTINCT_PREVIEW_VALUES <= 0:
    raise ValueError("Distinct profiling limits must be positive integers.")

def profile_distinct_values(input_path, output_dir, chunk_rows=10_000, memory_values_per_column=5_000,
                            preview_values=8, overwrite=True, show_progress=True):
    input_path, output_dir = Path(input_path), Path(output_dir)
    if min(chunk_rows, memory_values_per_column, preview_values) < 1:
        raise ValueError("Chunk size, memory threshold, and preview size must be positive.")
    if not input_path.is_file(): raise FileNotFoundError(f"Dataset not found: {input_path}")
    output_dir.mkdir(parents=True, exist_ok=True)
    summary_path = output_dir / f"{input_path.stem}_distinct_summary.csv"
    values_path = output_dir / f"{input_path.stem}_distinct_values.csv"
    if not overwrite and any(p.exists() for p in (summary_path, values_path)):
        raise FileExistsError("Distinct reports already exist; set DISTINCT_OVERWRITE=True to replace them.")
    started = time.perf_counter()
    with tempfile.TemporaryDirectory(prefix="afc_distinct_") as scratch:
        connection = sqlite3.connect(str(Path(scratch) / "counts.sqlite"))
        # These performance settings apply only to a disposable scratch database.
        connection.execute("PRAGMA journal_mode=OFF")
        connection.execute("PRAGMA synchronous=OFF")
        connection.execute("PRAGMA cache_size=-65536")
        connection.execute("CREATE TABLE counts (column_id INTEGER, value TEXT COLLATE BINARY, "
                           "occurrences INTEGER NOT NULL, PRIMARY KEY(column_id, value)) WITHOUT ROWID")
        temp_paths = []
        try:
            with input_path.open(encoding="utf-8-sig", newline="") as handle:
                reader = csv.reader(handle, strict=True)
                headers = next(reader, None)
                if not headers or len(headers) != len(set(headers)) or any(not h for h in headers):
                    raise ValueError("Dataset headers are empty or duplicated.")
                counters = [{} for _ in headers]
                spilled = set()
                total_rows = 0
                def flush_large_columns(final=False):
                    for column_id, counter in enumerate(counters):
                        if len(counter) > memory_values_per_column: spilled.add(column_id)
                        if column_id in spilled and counter:
                            connection.executemany(
                                "INSERT INTO counts VALUES (?, ?, ?) ON CONFLICT(column_id, value) "
                                "DO UPDATE SET occurrences=occurrences+excluded.occurrences",
                                ((column_id, value, count) for value, count in sorted(counter.items())))
                            counter.clear()
                    connection.commit()
                for row in reader:
                    if len(row) != len(headers):
                        raise ValueError(f"Wrong field count at CSV line {reader.line_num}")
                    for counter, value in zip(counters, row):
                        counter[value] = counter.get(value, 0) + 1
                    total_rows += 1
                    if total_rows % chunk_rows == 0: flush_large_columns()
                    if show_progress and total_rows % 100_000 == 0:
                        print(f"Distinct scan: {input_path.name}: {total_rows:,} rows; {time.perf_counter() - started:.1f}s", flush=True)
                flush_large_columns(final=True)
            for _ in range(2):
                descriptor, name = tempfile.mkstemp(dir=output_dir, suffix=".csv.tmp")
                os.close(descriptor)
                temp_paths.append(Path(name))
            result = []
            summary_headers = ["dataset", "column_position", "column_name", "distinct_count_including_blank",
                               "distinct_count_excluding_blank", "blank_row_count", "nonblank_row_count",
                               "total_rows", "distinct_preview_json"]
            with temp_paths[0].open("w", encoding="utf-8-sig", newline="") as summary_handle, \
                 temp_paths[1].open("w", encoding="utf-8-sig", newline="") as values_handle:
                summary_writer = csv.DictWriter(summary_handle, fieldnames=summary_headers)
                summary_writer.writeheader()
                values_writer = csv.writer(values_handle)
                values_writer.writerow(["column_position", "column_name", "value", "is_blank", "occurrence_count"])
                for column_id, column_name in enumerate(headers):
                    items = (connection.execute("SELECT value, occurrences FROM counts WHERE column_id=? ORDER BY value",
                                                (column_id,)) if column_id in spilled else sorted(counters[column_id].items()))
                    distinct_count = blank_count = frequency_sum = 0
                    preview = []
                    for value, count in items:
                        distinct_count += 1
                        frequency_sum += count
                        if value == "": blank_count = count
                        if len(preview) < preview_values: preview.append(value)
                        values_writer.writerow([column_id + 1, column_name, value, "true" if value == "" else "false", count])
                    if frequency_sum != total_rows:
                        raise AssertionError(f"Frequency reconciliation failed for {column_name}")
                    record = dict(zip(summary_headers, [input_path.name, column_id + 1, column_name, distinct_count,
                                  distinct_count - int(blank_count > 0), blank_count, total_rows - blank_count,
                                  total_rows, json.dumps(preview, ensure_ascii=False)]))
                    summary_writer.writerow(record)
                    result.append(record)
            os.replace(temp_paths[1], values_path)
            os.replace(temp_paths[0], summary_path)
        finally:
            connection.close()
            for temporary in temp_paths: temporary.unlink(missing_ok=True)
    if show_progress:
        print(f"Distinct profiling complete: {len(headers)} columns, {total_rows:,} rows, {time.perf_counter() - started:.1f}s")
        print(f"Summary: {summary_path}\nComplete distinct values: {values_path}")
    return {"input": str(input_path), "summary_file": str(summary_path), "values_file": str(values_path),
            "columns": result, "total_rows": total_rows}


In [ ]:
# Verify exact-string counting, blank handling, frequency totals, and disk spilling.
# Test files are temporary and are deleted automatically; source datasets are untouched.
with tempfile.TemporaryDirectory() as distinct_test_dir:
    distinct_test_path = Path(distinct_test_dir) / "distinct_checks.csv"
    distinct_test_rows = [
        ["001", "97", ""], ["001", "HH", "STREET"], ["1", "hh", "NOT FOUND"],
        [" 001", "97", ""], ["", "", "NULL"], ["001", "HH", "street"],
    ]
    with distinct_test_path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["identifier", "country", "city"])
        writer.writerows(distinct_test_rows)
    distinct_test_result = profile_distinct_values(distinct_test_path, distinct_test_dir,
                          chunk_rows=2, memory_values_per_column=2, show_progress=False)
    with Path(distinct_test_result["values_file"]).open(encoding="utf-8-sig", newline="") as handle:
        distinct_test_values = list(csv.DictReader(handle))
    for i, column in enumerate(distinct_test_result["columns"]):
        expected = {}
        for row in distinct_test_rows: expected[row[i]] = expected.get(row[i], 0) + 1
        observed = {row["value"]: int(row["occurrence_count"]) for row in distinct_test_values
                    if int(row["column_position"]) == i + 1}
        assert observed == expected
        assert column["distinct_count_including_blank"] == len(expected)
        assert column["blank_row_count"] == expected.get("", 0)
        assert column["distinct_count_excluding_blank"] == len(expected) - int("" in expected)
    assert distinct_test_result["total_rows"] == len(distinct_test_rows)
print("Distinct checks passed: raw text, leading zeros, case, whitespace, literal NULL, empty fields, and SQLite spill aggregation.")


In [ ]:
distinct_missing_inputs = [str(path) for path in DISTINCT_INPUTS if not path.is_file()]
if distinct_missing_inputs:
    raise FileNotFoundError("Generate both datasets first. Missing: " + "; ".join(distinct_missing_inputs))
distinct_results = []
for distinct_input in DISTINCT_INPUTS:
    distinct_results.append(profile_distinct_values(
        distinct_input, DISTINCT_OUTPUT_DIR, chunk_rows=DISTINCT_CHUNK_ROWS,
        memory_values_per_column=DISTINCT_MEMORY_VALUES_PER_COLUMN,
        preview_values=DISTINCT_PREVIEW_VALUES, overwrite=DISTINCT_OVERWRITE))

# Display every column's counts and a short preview. Complete values are in the reports.
for distinct_result in distinct_results:
    print("\nDATASET:", Path(distinct_result["input"]).name)
    print("COLUMN | DISTINCT INCLUDING BLANK | DISTINCT EXCLUDING BLANK | BLANK ROWS | VALUE PREVIEW")
    for column in distinct_result["columns"]:
        print(f"{column['column_position']:03d} {column['column_name']} | "
              f"{column['distinct_count_including_blank']:,} | {column['distinct_count_excluding_blank']:,} | "
              f"{column['blank_row_count']:,} | {column['distinct_preview_json']}")
print("Distinct reports completed for both datasets; source CSVs were not modified.")
